# Лабораторная работа №1
## «Предварительный анализ данных»

**ФИО:** Сокуров Олег Асланович  
**Группа:** 4415  
**Вариант:** 17

### Цель работы
Осуществить предварительную обработку данных CSV-файла, выявить и устранить проблемы в данных, а затем выполнить группировки и сводную таблицу в соответствии с вариантом.

### Набор данных
Используется `salary.csv`, содержащий сведения о зарплатах сотрудников: год выплаты, тип занятости, должность, зарплату в исходной валюте, зарплату в долларах, страны проживания и расположения компании, размер компании и показатели опыта.

## Вариант 17

1. Отфильтровать набор данных.
2. Выбрать TOP-7 должностей по средней зарплате в долларах.
3. Определить TOP-2 года выплаты заработной платы по количеству записей в году.
4. Для отфильтрованного набора данных создать сводную таблицу:
   - строки — `job_title`;
   - столбцы — `employment_type`;
   - значения — медианная зарплата в долларах;
   - показатель — `salary_in_usd`.

## 1. Загрузка данных

Файл содержит разделитель `;`, поэтому при чтении явно указывается параметр `sep=';'`. Это позволяет получить фактическую структуру из 10 столбцов.

In [1]:
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 45)
pd.set_option('display.width', 140)

file_path = 'salary.csv'
df = pd.read_csv(file_path, sep=';')

print('Размер исходного набора:', df.shape)
print('Столбцы:', df.columns.tolist())

Размер исходного набора: (401, 10)
Столбцы: ['work_year', 'employment_type', 'job_title', 'salary', 'salary_in_usd', 'employee_residence', 'company_location', 'company_size', 'exp_all', 'exp']


## 2. Первые 20 строк набора данных

Для первичного ознакомления выведем первые 20 записей с помощью `head(20)`. Это позволяет проверить фактический формат строк и содержимое столбцов.

In [2]:
display(df.head(20))

,work_year,employment_type,job_title,salary,salary_in_usd,employee_residence,company_location,company_size,exp_all,exp
0,2020.0,FT,Data SCIENTIST,70000.0,79833.0,DE,DE,L,9,4
1,2020.0,FT,Product Data Analyst,20000.0,20000.0,HN,HN,S,2,2
2,2020.0,FT,Data Analyst,72000.0,72000.0,US,US,L,13,3
3,2020.0,FT,Data Scientist,11000000.0,35735.0,HU,HU,L,60,6
4,2020.0,FT,Data Scientist,45000.0,51321.0,FR,FR,S,8,4
5,2020.0,FT,Data Scientist,3000000.0,40481.0,IN,IN,L,53,1
6,2020.0,FT,Data Scientist,35000.0,39916.0,FR,FR,M,6,2
7,2020.0,FT,Data Analyst,85000.0,85000.0,US,US,L,15,2
8,2020.0,FT,Data Analyst,8000.0,8000.0,PK,PK,Large,1,4
9,2020.0,FT,Data Engineer,4450000.0,41689.0,JP,JP,S,50,5


## 3. Обзор данных и предметной области

Набор данных относится к предметной области анализа заработных плат. В каждой строке содержится информация о сотруднике или записи о его зарплате за определённый год.

In [3]:
column_description = pd.DataFrame({
    'Столбец': df.columns,
    'Тип': df.dtypes.astype(str).values,
    'Назначение': [
        'Год выплаты заработной платы',
        'Тип занятости: FT — полная занятость, PT — частичная, FL — фриланс',
        'Название должности',
        'Годовая зарплата в исходной валюте',
        'Годовая зарплата в долларах США',
        'Страна проживания сотрудника',
        'Страна расположения головного офиса компании',
        'Размер компании',
        'Общий опыт работы',
        'Показатель опыта из исходного набора'
    ]
})
display(column_description)

,Столбец,Тип,Назначение
0,work_year,float64,Год выплаты заработной платы
1,employment_type,object,"Тип занятости: FT — полная занятость, PT ..."
2,job_title,object,Название должности
3,salary,float64,Годовая зарплата в исходной валюте
4,salary_in_usd,float64,Годовая зарплата в долларах США
5,employee_residence,object,Страна проживания сотрудника
6,company_location,object,Страна расположения головного офиса компании
7,company_size,object,Размер компании
8,exp_all,int64,Общий опыт работы
9,exp,int64,Показатель опыта из исходного набора


## 4. Исследование структуры данных

Метод `info()` используется для проверки количества строк и столбцов, типов данных и количества заполненных значений.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 401 entries, 0 to 400
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   work_year           401 non-null    float64
 1   employment_type     401 non-null    object 
 2   job_title           401 non-null    object 
 3   salary              398 non-null    float64
 4   salary_in_usd       401 non-null    float64
 5   employee_residence  401 non-null    object 
 6   company_location    401 non-null    object 
 7   company_size        401 non-null    object 
 8   exp_all             401 non-null    int64  
 9   exp                 401 non-null    int64  
dtypes: float64(3), int64(2), object(5)
memory usage: 31.5+ KB


## 5. Оценка числовых столбцов

Для числовых признаков применён метод `describe()`. Он показывает объём выборки, среднее значение, стандартное отклонение, квартили, минимум и максимум.

In [5]:
display(df.describe())

,work_year,salary,salary_in_usd,exp_all,exp
count,401.000000,3.980000e+02,401.000000,401.000000,401.000000
mean,2021.528678,2.888336e+05,105895.017456,17.197007,4.486284
std,0.678086,1.677081e+06,58183.664171,11.133445,2.027421
min,2020.000000,4.000000e+03,2859.000000,0.000000,-1.000000
25%,2021.000000,6.700000e+04,65013.000000,10.000000,3.000000
50%,2022.000000,1.091400e+05,100000.000000,15.000000,4.000000
75%,2022.000000,1.500000e+05,140000.000000,22.000000,6.000000
max,2022.000000,3.040000e+07,412000.000000,81.000000,10.000000


## 6. Проверка названий столбцов

Выведем названия столбцов через `df.columns` и проверим их на пробелы по краям и другие очевидные проблемы. Названия уже приведены к формату `snake_case`, поэтому переименование самих столбцов не требуется.

In [6]:
print('Названия столбцов:')
display(pd.DataFrame({'Столбец': df.columns}))

print('Проблемы с названиями:')
print('Пробелы по краям:', any(col != col.strip() for col in df.columns))
print('Пустые названия:', any(col == '' for col in df.columns))
print('Повторы названий:', df.columns.duplicated().any())


Названия столбцов:


,Столбец
0,work_year
1,employment_type
2,job_title
3,salary
4,salary_in_usd
5,employee_residence
6,company_location
7,company_size
8,exp_all
9,exp


Проблемы с названиями:
Пробелы по краям: False
Пустые названия: False
Повторы названий: False


## 7. Проверка и обработка пропусков

По результатам проверки пропуски присутствуют только в `salary` — 3 значения. В `salary_in_usd`, используемом в задании варианта 17, пропусков нет. Согласно методическим указаниям количественные пропуски можно заменить медианой.

Поэтому значения `salary` заменены медианой `109140`. Строки при этом не удаляются, а целевые показатели варианта 17 не меняются, поскольку расчёты выполняются по `salary_in_usd`.

In [7]:
missing_before = df.isna().sum()
print('Пропуски до обработки:')
display(missing_before.to_frame('Количество'))

salary_median = df['salary'].median()
df['salary'] = df['salary'].fillna(salary_median)

print(f'Медиана salary для замены пропусков: {salary_median:.0f}')
print('\nПропуски после обработки:')
display(df.isna().sum().to_frame('Количество'))

Пропуски до обработки:


,Количество
work_year,0
employment_type,0
job_title,0
salary,3
salary_in_usd,0
employee_residence,0
company_location,0
company_size,0
exp_all,0
exp,0


Медиана salary для замены пропусков: 109140

Пропуски после обработки:


,Количество
work_year,0
employment_type,0
job_title,0
salary,0
salary_in_usd,0
employee_residence,0
company_location,0
company_size,0
exp_all,0
exp,0


## 8. Проверка явных и неявных дубликатов

Полные дубликаты строк проверяются методом `duplicated()`. Неявные дубликаты определены среди категориальных значений: варианты `Data SCIENTIST`, `DataScientist` и `Data Scientist` обозначают одну должность с различиями в регистре/написании; аналогично `Data AnalyticsManager` и `Data Analytics Manager`. Для `company_size` значение `Large` соответствует категории `L`.

Эти альтернативные написания исправляются методом `replace()`, как рекомендовано в методическом материале.

In [8]:
print('Количество полных дубликатов до исправления:', int(df.duplicated().sum()))

print('\nУникальные значения job_title до исправления:')
display(pd.Series(sorted(df['job_title'].unique()), name='job_title'))

job_title_map = {
    'Data SCIENTIST': 'Data Scientist',
    'DataScientist': 'Data Scientist',
    'Data AnalyticsManager': 'Data Analytics Manager'
}
company_size_map = {'Large': 'L'}

df['job_title'] = df['job_title'].replace(job_title_map)
df['company_size'] = df['company_size'].replace(company_size_map)

print('\nКоличество полных дубликатов после исправления:', int(df.duplicated().sum()))
print('\nИсправленные категории job_title:')
for old, new in job_title_map.items():
    print(f'{old!r} -> {new!r}')
print('\nИсправленные категории company_size:')
print("'Large' -> 'L'")

Количество полных дубликатов до исправления: 0

Уникальные значения job_title до исправления:


0                   Data Analyst
1        Data Analytics Engineer
2            Data Analytics Lead
3         Data Analytics Manager
4          Data AnalyticsManager
5                  Data Engineer
6                 Data SCIENTIST
7          Data Science Engineer
8                 Data Scientist
9                  DataScientist
10          Head of Data Science
11      Head of Machine Learning
12    Machine Learning Developer
13      Machine Learning Manager
14                  NLP Engineer
15          Product Data Analyst
Name: job_title, dtype: object


Количество полных дубликатов после исправления: 0

Исправленные категории job_title:
'Data SCIENTIST' -> 'Data Scientist'
'DataScientist' -> 'Data Scientist'
'Data AnalyticsManager' -> 'Data Analytics Manager'

Исправленные категории company_size:
'Large' -> 'L'


## 9. Проверка типов данных

Тип `work_year` после чтения имеет вид `float64`, хотя это год и он должен храниться как целое число. После проверки отсутствия пропусков столбец переводится в `int`. Остальные типы соответствуют содержимому и задачам анализа.

In [9]:
print('Типы до преобразования:')
display(df.dtypes.to_frame('dtype'))

df['work_year'] = df['work_year'].astype(int)

print('\nТипы после преобразования work_year:')
display(df.dtypes.to_frame('dtype'))

Типы до преобразования:


,dtype
work_year,float64
employment_type,object
job_title,object
salary,float64
salary_in_usd,float64
employee_residence,object
company_location,object
company_size,object
exp_all,int64
exp,int64



Типы после преобразования work_year:


,dtype
work_year,int64
employment_type,object
job_title,object
salary,float64
salary_in_usd,float64
employee_residence,object
company_location,object
company_size,object
exp_all,int64
exp,int64


## 10. Задание варианта 17 — подготовленный набор данных

После предварительной обработки сохраняется 401 строка и 10 столбцов. Подготовленный датасет также сохраняется в отдельный CSV-файл.

In [10]:
print('Размер подготовленного набора:', df.shape)
print('Пропуски:', int(df.isna().sum().sum()))
print('Полные дубликаты:', int(df.duplicated().sum()))

df.to_csv('salary_variant17_preprocessed.csv', sep=';', index=False)
print('Файл salary_variant17_preprocessed.csv сохранён.')

Размер подготовленного набора: (401, 10)
Пропуски: 0
Полные дубликаты: 0
Файл salary_variant17_preprocessed.csv сохранён.


## 11. TOP-7 должностей по средней зарплате

Средняя зарплата `salary_in_usd` рассчитывается по каждой должности с помощью `groupby()` и `mean()`. Далее результаты сортируются по убыванию, после чего программно выбираются первые семь должностей.

In [11]:
avg_salary_by_job = (
    df.groupby('job_title', as_index=True)['salary_in_usd']
      .mean()
      .sort_values(ascending=False)
)

top7_jobs = avg_salary_by_job.head(7)

print('Средняя зарплата по всем должностям:')
display(avg_salary_by_job.round(2).rename('Средняя зарплата, USD').to_frame())

print('\nTOP-7 должностей:')
display(top7_jobs.round(2).rename('Средняя зарплата, USD').to_frame())

Средняя зарплата по всем должностям:


,"Средняя зарплата, USD"
job_title,
Data Analytics Lead,405000.00
Head of Data Science,146718.75
Data Analytics Manager,130025.00
Machine Learning Manager,117104.00
Data Engineer,112725.00
Data Scientist,108187.83
Data Analyst,92628.85
Machine Learning Developer,85860.67
Head of Machine Learning,79039.00



TOP-7 должностей:


,"Средняя зарплата, USD"
job_title,
Data Analytics Lead,405000.00
Head of Data Science,146718.75
Data Analytics Manager,130025.00
Machine Learning Manager,117104.00
Data Engineer,112725.00
Data Scientist,108187.83
Data Analyst,92628.85


## 12. TOP-2 года по количеству записей

Количество записей для каждого `work_year` считается методом `value_counts()`. После сортировки по убыванию программно выбираются два года с наибольшим числом записей.

In [12]:
records_by_year = df['work_year'].value_counts().sort_values(ascending=False)
top2_years = records_by_year.head(2)

print('Количество записей по годам:')
display(records_by_year.rename('Количество записей').to_frame())

print('\nTOP-2 года:')
display(top2_years.rename('Количество записей').to_frame())

Количество записей по годам:


,Количество записей
work_year,
2022,254
2021,105
2020,42



TOP-2 года:


,Количество записей
work_year,
2022,254
2021,105


## 13. Фильтрация исходного набора

В отфильтрованный набор входят одновременно только TOP-7 должностей и TOP-2 года, полученные программно на предыдущих этапах.

In [13]:
filtered_df = df[
    df['job_title'].isin(top7_jobs.index) &
    df['work_year'].isin(top2_years.index)
].copy()

print('Размер отфильтрованного набора:', filtered_df.shape)
print('\nКоличество записей по годам в фильтре:')
display(filtered_df['work_year'].value_counts().sort_index().rename('Количество записей').to_frame())

print('\nПервые 15 строк отфильтрованного набора:')
display(filtered_df.head(15))

Размер отфильтрованного набора: (347, 10)

Количество записей по годам в фильтре:


,Количество записей
work_year,
2021,99
2022,248



Первые 15 строк отфильтрованного набора:


,work_year,employment_type,job_title,salary,salary_in_usd,employee_residence,company_location,company_size,exp_all,exp
42,2021,FT,Data Scientist,45000.0,53192.0,FR,FR,L,6,1
43,2021,FT,Data Analyst,80000.0,80000.0,US,US,M,11,3
45,2021,FT,Data Engineer,140000.0,140000.0,US,US,L,17,3
46,2021,FT,Data Engineer,110000.0,28476.0,PL,PL,L,14,7
47,2021,FT,Data Analyst,50000.0,59102.0,FR,FR,M,7,4
49,2021,FT,Data Analyst,80000.0,80000.0,BG,US,S,10,7
50,2021,FT,Data Scientist,2200000.0,29751.0,IN,IN,L,27,9
51,2021,FT,Data Analyst,75000.0,75000.0,US,US,L,10,4
52,2021,FT,Data Engineer,150000.0,150000.0,US,US,L,27,4
53,2021,FT,Data Analyst,62000.0,62000.0,US,US,L,8,5


## 14. Итоговая сводная таблица

Создаётся `pivot_table` с параметрами `index='job_title'`, `columns='employment_type'`, `values='salary_in_usd'` и `aggfunc='median'`. Для полноты отображения все семь выбранных должностей возвращаются в исходном порядке TOP-7; если для должности после фильтрации нет записей, в соответствующих ячейках остаётся `NaN`.

Строка показывает должность, столбцы — тип занятости (`FL`, `FT`, `PT`), а значение ячейки — медианную зарплату в долларах по данной комбинации.

In [14]:
pivot_result = pd.pivot_table(
    filtered_df,
    index='job_title',
    columns='employment_type',
    values='salary_in_usd',
    aggfunc='median'
)

pivot_result = pivot_result.reindex(index=top7_jobs.index)
pivot_result = pivot_result.reindex(columns=['FL', 'FT', 'PT'])

print('Итоговая сводная таблица (медианная зарплата, USD):')
display(pivot_result.round(2))

Итоговая сводная таблица (медианная зарплата, USD):


employment_type,FL,FT,PT
job_title,,,
Data Analytics Lead,NaN,405000.0,NaN
Head of Data Science,NaN,138937.5,NaN
Data Analytics Manager,NaN,130000.0,NaN
Machine Learning Manager,NaN,NaN,NaN
Data Engineer,20000.0,110250.0,62349.0
Data Scientist,100000.0,117351.5,100000.0
Data Analyst,NaN,96350.0,10354.0


## 15. Содержательный анализ и вывод

После очистки данных в наборе осталось 401 наблюдение. В столбце `salary` первоначально было 3 пропуска; они заменены медианой 109140. Полных дубликатов не обнаружено. Исправлены неявные дубликаты в названиях должностей и размере компании, а `work_year` приведён к целочисленному типу.

По средней зарплате TOP-7 после нормализации названий образуют: Data Analytics Lead, Head of Data Science, Data Analytics Manager, Machine Learning Manager, Data Engineer, Data Scientist и Data Analyst. По количеству записей лидируют 2022 год (254 записи) и 2021 год (105 записей). Для фильтрации остаётся 347 строк.

В сводной таблице основная масса наблюдений относится к полной занятости FT. Для Machine Learning Manager после ограничения двумя наиболее представленными годами записей нет, поэтому значения в строке отсутствуют. Наиболее высокая медианная зарплата в итоговой таблице наблюдается у Data Analytics Lead при FT — 405000 USD. Для Data Engineer доступны все три типа занятости: FL, FT и PT.